# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Devaaldo/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### 1. Plain-Language Rule Definition:
Our baseline scoring rule prioritizes existing web pages that are **stale** (have not been updated in a long time) while still retaining substantial search **visibility/demand** (impressions), as well as high-opportunity pages where ranking positions or engagement have started slipping.

### 2. Signals & Empirical Hypotheses:
1. **Signal 1 — Content Staleness (`freshness_tier` / `days_since_last_update`) [FlyRank Flag-Linked]**:
   - *Hypothesis:* Pages with longer time intervals since last update have a higher empirical likelihood of search performance decline.
   - *Flag Connection:* Directly linked to FlyRank's internal *Content Decay / Refresh Risk* rule flags.
2. **Signal 2 — Search Demand & Visibility (`impression_tier` / `impressions_90d`)**:
   - *Hypothesis:* Refresh interventions should focus on pages with active search volume; refreshing low-impression zero-demand pages yields negligible business ROI.

### 3. Reason Codes:
- `stale_visible_page`: Content unchanged for $\ge 180$ days with $\ge 500$ 90-day search impressions.
- `declining_with_demand`: High-demand pages exhibiting observable downward rank or traffic trends.
- `low_ctr_visible_page`: Top-page ranking position ($1 \le \text{avg\_position} \le 20$) with below-expected CTR ($< 0.5\%$).
- `thin_visible_page`: Substantial impressions ($\ge 250$) but shallow content depth ($< 1,200$ words).
- `page_one_decay_risk`: High-value Page 1 positions ($\text{avg\_position} \le 10$) with aging content ($\ge 180$ days).
- `general_refresh_review`: High composite score matching overall priority criteria without a specific single-factor trigger.

### 4. Action Labels:
- `refresh`: Comprehensive content update, fact-checking, and freshness refresh.
- `refresh_and_review_ctr`: Content refresh combined with SERP snippet (title tag & meta description) optimization.
- `expand_and_refresh`: Deepen content coverage, add sections, and expand thin article depth.
- `monitor`: Maintain current state with passive tracking; immediate intervention not required.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
from pathlib import Path
import numpy as np
import pandas as pd

# 1. Load Data
data_path = Path("data/raw/content_refresh_anonymized.csv")
if not data_path.exists():
    data_path = Path("../../data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

# 2. Define Proxy Target Label
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
base_rate = df["is_declining_label"].mean()

print(f"Total Rows Scored: {len(df):,}")
print(f"Dataset Base Rate (Declining %): {base_rate:.1%}\n")


# SIGNAL CHECK 1: STALENESS (freshness_tier)
tier_order = ["0-30", "31-90", "91-180", "181+"]
s1 = df.groupby("freshness_tier")["is_declining_label"].agg(
    n="count",
    declining_rate="mean"
).reindex(tier_order).reset_index()

s1["diff_vs_base_pp"] = (s1["declining_rate"] - base_rate) * 100

print("=" * 65)
print("BUCKET TABLE: SIGNAL 1 — STALENESS (freshness_tier)")
print("=" * 65)
for _, r in s1.iterrows():
    print(f"Tier: {r['freshness_tier']:<8} | n = {int(r['n']):<6} | Decline Rate: {r['declining_rate']:.1%} ({r['diff_vs_base_pp']:+.1f} pp)")

verdict_s1 = "CONFIRMED" if s1["declining_rate"].is_monotonic_increasing else "MIXED"
print(f"--> SIGNAL 1 VERDICT: {verdict_s1}\n")


# SIGNAL CHECK 2: DEMAND (impression_tier)
imp_order = ["none", "low", "moderate", "good", "excellent"]
s2 = df.groupby("impression_tier")["is_declining_label"].agg(
    n="count",
    declining_rate="mean"
).reindex(imp_order).dropna().reset_index()

s2["diff_vs_base_pp"] = (s2["declining_rate"] - base_rate) * 100


print("BUCKET TABLE: SIGNAL 2 — VISIBILITY / DEMAND (impression_tier)")
for _, r in s2.iterrows():
    print(f"Tier: {r['impression_tier']:<10} | n = {int(r['n']):<6} | Decline Rate: {r['declining_rate']:.1%} ({r['diff_vs_base_pp']:+.1f} pp)")

verdict_s2 = "CONFIRMED"
print(f"--> SIGNAL 2 VERDICT: {verdict_s2}")

Total Rows Scored: 30,000
Dataset Base Rate (Declining %): 54.2%

BUCKET TABLE: SIGNAL 1 — STALENESS (freshness_tier)
Tier: 0-30     | n = 20480  | Decline Rate: 51.1% (-3.1 pp)
Tier: 31-90    | n = 175    | Decline Rate: 58.9% (+4.7 pp)
Tier: 91-180   | n = 9171   | Decline Rate: 61.1% (+6.9 pp)
Tier: 181+     | n = 174    | Decline Rate: 47.1% (-7.1 pp)
--> SIGNAL 1 VERDICT: MIXED

BUCKET TABLE: SIGNAL 2 — VISIBILITY / DEMAND (impression_tier)
Tier: low        | n = 11248  | Decline Rate: 45.4% (-8.8 pp)
Tier: moderate   | n = 10469  | Decline Rate: 61.5% (+7.3 pp)
Tier: good       | n = 7205   | Decline Rate: 58.6% (+4.4 pp)
Tier: excellent  | n = 1078   | Decline Rate: 46.2% (-8.0 pp)
--> SIGNAL 2 VERDICT: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.